# Summary

## Tình huống

1.  Ứng Dụng: GrabMart
    - Chương trình khuyến mãi: Cross-sell mega promo (chương trình khuyến mãi chéo quy mô lớn)
    - Khoảng tgian thi triển: 6 tuần
    - Địa diểm: Thành Phố Hồ Chí Minh
    - Trạng Thái: thí điểm

2. Merchanism:
    Khi đặt đơn hàng trên `GrabFood`:
    constraints:
        - tổng giá trị > 100.000 vnđ -> nhận voucher trị giá 50.000 vnđ cho `GrabMart` tiếp theo

3. Output - Backlog Test:
    - Tổng giá trị giao dịch (GMV):
        + GrabMart và GrabFood: đều tăng hơn 45% (so với ?)

4. Deployment:
    - Triển khai chính thức tại Hà Nội (với ngân sách khủng)

5. Work-to-do: đưa ra đánh giá -- và phân tích răng có hợp lí để deploy ở Hà Nội không ?

    Tình huống:

        Giám đốc Quốc gia tin rằng chiến dịch này đã thành công trong việc thay đổi hành vi người dùng và 
        hình thành thói quen sử dụng đa dịch vụ.

    Câu hỏi:
        
        - Dựa trên dữ liệu chuyên sâu về hành vi và kết quả kinh doanh (P&L), bạn có đồng ý với nhận định này không?
        - Yếu tố thực sự nào đang thúc đẩy mức tăng trưởng 45% đó? 
        - Hãy xây dựng lập luận dựa trên dữ liệu và đưa ra khuyến nghị cuối cùng về việc có nên mở rộng chương trình này sang Hà Nội hay không 

## Import Libraries

In [11]:
import sqlite3
import pandas as pd

## Create Connection to Database

In [4]:
conn = sqlite3.connect("grab_vn_case_analytics.db")

## Analytics view

### View table available

In [7]:
query = """
SELECT name, sql
FROM sqlite_master
WHERE type = 'table';
"""

for name, sql in conn.execute(query):
    print(name)
    print(sql)

users
CREATE TABLE "users" (
"user_id" TEXT,
  "city" TEXT,
  "user_segment" TEXT,
  "first_mart_order_date" TEXT
)
merchants
CREATE TABLE "merchants" (
"merchant_id" TEXT,
  "merchant_name" TEXT,
  "vertical" TEXT,
  "merchant_type" TEXT,
  "take_rate" REAL
)
weather_daily
CREATE TABLE "weather_daily" (
"date" TEXT,
  "city" TEXT,
  "avg_temperature_celsius" REAL
)
transactions
CREATE TABLE "transactions" (
"transaction_id" TEXT,
  "date" TEXT,
  "week_number" INTEGER,
  "period" TEXT,
  "city" TEXT,
  "service_type" TEXT,
  "user_id" TEXT,
  "merchant_id" TEXT,
  "item_subtotal" REAL,
  "delivery_fee_collected" REAL,
  "driver_payout" REAL,
  "promo_discount" REAL,
  "is_cross_sell_voucher" INTEGER
)


### Prepare dataframe

In [15]:
query_transactions="""select * from transactions"""
query_users = """select * from users"""
query_merchants = """select * from merchants"""
query_weather_daily = """select * from weather_daily"""

In [16]:
df_transactions = pd.read_sql(
    sql=query_transactions,
    con=conn
)
df_users = pd.read_sql(
    sql=query_users,
    con=conn
)
df_merchants = pd.read_sql(
    sql=query_merchants,
    con=conn
)
df_weather_daily = pd.read_sql(
    sql=query_weather_daily,
    con=conn
)

### Deep-Analysis -- with query

In [51]:
class SQLite3Connection:
    def __init__(
        self,
        db_path:str
    ):
        self.db_path = db_path
        self.conn = sqlite3.connect(self.db_path)

    @staticmethod
    def parse_query(
        query_path:str
    ) -> str:
        with open(query_path, 'r') as query:
            return query.read()

    def query_to_dataframe(
        self,
        query:str
    ) -> pd.DataFrame:
        cursor = self.conn.execute(query)
        data = cursor.fetchall()
        header = [col[0] for col in cursor.description]

        return pd.DataFrame(data=data, columns=header)

In [52]:
data = SQLite3Connection(db_path="grab_vn_case_analytics.db")

## Background-Analysis

## In-Promotion Duration

### Acquisition Funnel

**Hypothesis:**

`45% GMV growth` của Food có thể được campaign kích hoạt từ existing Food users -- vì mechanic bắt đầu từ một `Food order >100k`

```mermaid
mermaid TD
    A[Existing GrabFood user] -->|Food order > 100k| B[Eligible for voucher]
    B -->|50k voucher| C[GrabMart voucher redemption]
    C --> D[GrabMart transaction]
```

| Metric                            | Ý nghĩa                                                 |
| --------------------------------- | ------------------------------------------------------- |
| **Food eligibility rate**         | % Food users/orders đủ điều kiện                        |
| **Voucher redemption rate**       | % eligible users thực sự redeem voucher                 |
| **Mart acquisition rate**         | % Food users được campaign chuyển thành Mart users      |
| **Voucher-driven Mart GMV share** | Mart GMV đến từ transaction có voucher / total Mart GMV |
| **Voucher-driven GMV uplift**     | incremental GMV nếu có thể establish baseline/control   |


_result:_ 



| |metric | value |
--|-------|-------|
0|food_users|4999.00|
1|eligible_food_users|4991.00|
2|voucher_redeemers|4446.00|
3|campaign_acquired_users|4441.00|
4|food_eligibility_rate|99.84|
5|voucher_redemption_rate|89.08|
6|campaign_acquisition_rate|88.84|





In [83]:
query=data.parse_query(query_path="query/acquisition_funnel.sql")
df = data.query_to_dataframe(query=query)
value_arr = df.columns
df_pivoted = df.melt(
    value_vars=value_arr,
    var_name="metric",
    value_name="value"
)

### % voucher-driven transaction int Total-GMV

>Trong 45% growth của GMV (both Mart/Food) thì bao nhiêu phần trăm đến từ voucher của Mart

`sql_path`: "query/percentage_voucher_driven_transaction.sql"

_result_:


| |metric|value|
|-|------|-----|
0|total_mart_gmv|1792132.50
1|voucher_mart_gmv|781755.20
2|voucher_gmv_share|43.62





In [82]:
query=data.parse_query(query_path="query/percentage_voucher_driven_transaction.sql")
df=data.query_to_dataframe(query=query)
value_arr = df.columns
df_pivoted = df.melt(
    value_vars=value_arr,
    var_name="metric",
    value_name="value"
)

### Acquisition-Funnel (Deeper Layer)

>Trong số user/GMV của GrabMart trong Promo period, campaign thực sự mang vào bao nhiêu user mới so với những người vốn đã là Mart user?

Structure song song:

```
Promo-period GrabMart users
│
├── Existing Mart User
│   └── đã từng dùng Mart trước campaign
│
└── New Mart User
    └── first_mart_order_date bắt đầu trong/đúng thời điểm promo
```
sau đó -- `New-Mart` sẽ tiếp tục spill:
```
New Mart Users
│
├── Cross-sell acquired
│   └── is_cross_sell_voucher = 1
│
└── Non-cross-sell
```
`sql_path`: "query/acquisition_funnel_inc_user_type.sql"


__Result__:


| mart_user_type | users | user_share | cross_sell_users | cross_sell_user_rate | mart_gmv | gmv_share | cross_sell_gmv | cross_sell_gmv_share | gmv_per_user |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| New Mart User | 643 | 13.24 | 643 | 100.00 | 240,164.6 | 13.4 | 111,391.1 | 46.38 | 373.51 |
| Existing Mart User | 4,215 | 86.76 | 3,804 | 90.25 | 1,551,967.9 | 86.6 | 670,364.1 | 43.19 | 368.20 |

In [88]:
query = data.parse_query(query_path="query/acquisition_funnel_inc_user_type.sql")
data.query_to_dataframe(query=query)


,mart_user_type,users,user_share,cross_sell_users,cross_sell_user_rate,mart_gmv,gmv_share,cross_sell_gmv,cross_sell_gmv_share,gmv_per_user
0,New Mart User,643,13.24,643,100.00,240164.6,13.4,111391.1,46.38,373.51
1,Existing Mart User,4215,86.76,3804,90.25,1551967.9,86.6,670364.1,43.19,368.20


### Query-base

### Check-whether: in Promo Duration -- is there any user | not aware about the promotion

_no_used_any_transaction_in_promo_duration_

`sql_path`: query/user_no_aware_promotion.sql

user_segment | cnt_user | Food_users | Mart_users
-------------|----------|------------|-----------
High Value|29|29|29
Occasional|288|288|167
Regular|168|168|152
Voucher Hunter|68|68|63

In [55]:
query=data.parse_query(query_path="query/user_no_aware_promotion.sql")
data.query_to_dataframe(query=query)

,user_segment,cnt_user,Food_users,Mart_users
0,High Value,29,29,29
1,Occasional,288,288,167
2,Regular,168,168,152
3,Voucher Hunter,68,68,63


### Segment (existed | new) in each duration (Before-Promo | Promo | After-Promo) 

`sql_path`: "query/classfiy_users_new_new_or_exists.sql"

||period|user_type|unique_users|n_transactions
|-|------|---------|------------|--------------
|0|Post-Promo|existed-mart-user|2634|4225
|1|Pre-Promo|existed-mart-user|3984|12543
|2|Promo|existed-mart-user|4215|20420
|3|Post-Promo|new-mart-user|385|655
|4|Pre-Promo|new-mart-user|598|1974
|5|Promo|new-mart-user|643|3210

In [61]:
query=data.parse_query(query_path="query/classify_users_new_or_exist.sql")
data.query_to_dataframe(query=query)

,period,user_type,unique_users,n_transactions
0,Post-Promo,existed-mart-user,2634,4225
1,Pre-Promo,existed-mart-user,3984,12543
2,Promo,existed-mart-user,4215,20420
3,Post-Promo,new-mart-user,385,655
4,Pre-Promo,new-mart-user,598,1974
5,Promo,new-mart-user,643,3210


### who actual drove the GROWTH ?

In [69]:
query=data.parse_query(query_path="query/segment_drive_growth.sql")
data.query_to_dataframe(query=query)

,city,user_segment,user_type,gross_gmv_M_vnd,service_type
0,Ho Chi Minh City,High Value,existed-mart-user,4472.2,GrabFood
1,Ho Chi Minh City,Regular,existed-mart-user,2950.7,GrabFood
2,Ho Chi Minh City,Voucher Hunter,existed-mart-user,1558.7,GrabFood
3,Ho Chi Minh City,High Value,new-mart-user,795.6,GrabFood
4,Ho Chi Minh City,Regular,existed-mart-user,734.4,GrabMart
5,Ho Chi Minh City,Occasional,existed-mart-user,509.7,GrabFood
6,Ho Chi Minh City,Regular,new-mart-user,441.0,GrabFood
7,Ho Chi Minh City,High Value,existed-mart-user,406.7,GrabMart
8,Ho Chi Minh City,Voucher Hunter,existed-mart-user,318.7,GrabMart
9,Ho Chi Minh City,Voucher Hunter,new-mart-user,254.0,GrabFood


### Check PnL: structure

In [35]:
query = """
select
city,
user_segment,
count(user_id) as cnt_user
from users
where 1=1
group by
    city,
    user_segment
"""
data.query_to_dataframe(query=query)

,city,user_segment,cnt_user
0,Da Nang,High Value,408
1,Da Nang,Occasional,307
2,Da Nang,Regular,919
3,Da Nang,Voucher Hunter,366
4,Hanoi,High Value,738
5,Hanoi,Occasional,489
6,Hanoi,Regular,1604
7,Hanoi,Voucher Hunter,669
8,Ho Chi Minh City,High Value,984
9,Ho Chi Minh City,Occasional,730
